In [1]:
from radpy.stellar import *
from radpy.sedfit import *

/home/oxfor/miniforge/envs/test_env/lib/python3.12/site-packages/radpy/sedfit.py:11: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Holoviews not imported. Some visualizations will not be available.
PyMultiNest not imported.  MultiNest fits will not work.
/home/oxfor/miniforge/envs/test_env/lib/python3.12/site-packages/astroARIADNE/fitter.py:43: UserWarning: (py)MultiNest installation (or libmultinest.dylib) not detected.
  warnings.warn(


Checking existing file to see if MD5 sum matches ...
File exists. Not overwriting.
Checking existing file to see if MD5 sum matches ...
File exists. Not overwriting.


In [2]:
starname = 'HD 1461'
star = StellarParams()
filename = "/mnt/c/Users/oxfor/Research/rsadpy/tests/test_data/photometry/HD1461mags.dat"
phot_data = read_in_photometry(filename)
star.logg = 4.31
star.logg_err = 0.2155
star.feh = 0.19
star.feh_err = 0.01
ra_deg, dec_deg, ra_hms, dec_hms = pull_coords(starname, star, verbose = True)
star.dist, star.dist_err = distances(starname, verbose = True)


Coordinates in decimal degrees for HD 1461:
RA: 4.674448649133749, Dec: -8.053001063860556

Coordinates in sexagesimal format for HD 1461:
RA: 0:18:41.86768, Dec: -8:03:10.80383
Found Gaia DR3 ID: Gaia DR3 2430102808294101760
Corrected parallax: 42.6953 [mas]
Distance: 23.42178 +/- 0.01638 [pc]


In [ ]:
import numpy as np
def randomize_photometry(sed):
    idx = sed['index']
    filt = sed['sed_filter']
    wl = sed['la']
    width = sed['width']
    f = sed['flux']
    df = sed['eflux']

    #rand_wl = np.random.normal(wl, width)
    rand_f = np.random.normal(f, df)

    new_phot = pd.DataFrame(
        {'index': idx, 'sed_filter': filt, 'la': wl, 'width': width, 'flux': rand_f,
         'eflux': df})
    new_sed = Table.from_pandas(new_phot)
    new_sed['la'].unit = u.AA
    new_sed['width'].unit = u.AA
    new_sed['flux'] = rand_f
    new_sed['eflux'] = df
    new_sed['flux'].unit = u.erg / u.s / (u.cm ** 2) / u.AA
    new_sed['eflux'].unit = u.erg / u.s / (u.cm ** 2) / u.AA

    #if verbose:
    #    new_sed

    return new_sed

In [ ]:
teff = 5000
logg = 4.14
feh = 0.14
av = 0
model = 'phoenix'
sed = phot_data
num_iter = 5
fitT = True
fit_feh = False
fit_logg = False
fit_av = False
unit = 'AA'

In [ ]:
d = star.dist
ra = star.ra_hms
dec = star.dec_dms

f = io.StringIO()
with contextlib.redirect_stdout(f):
    x = SEDFit(ra, dec, 1,use_gaia_params = False, use_gaia_xp = False, grid_type=model)

x.dist = d
#teff, logg, feh, av = initial_guess
x.addguesses(teff=teff, logg=logg, feh=feh, av = av)
x.addrange(teff = [4000,8000])

downloadflux(x, sed)
set_quality(x)
#f1 = io.StringIO()
#with contextlib.redirect_stdout(f1):
#    x.fit(use_gaia=False, idx=np.arange(0, len(x.sed['index'])), fitdist=False, fitteff=fitT, fitfeh=fit_feh, 
#                fitlogg=fit_logg, fitav = fit_av, quality_check=False)

In [ ]:
x.sed

In [ ]:
rand_sed = randomize_photometry(sed)
downloadflux(x, rand_sed)

In [ ]:
x.sed

In [ ]:
_, _, _, _, model_w, model_f, _ = convert(x, unit=unit)

In [ ]:
F1 = np.trapz(model_f, model_w)

In [ ]:
F1

In [ ]:
# testing the integration convergence
wav_fine = np.linspace(model_w.min(), model_w.max(), 100*len(model_w))
#f_fine = interp_model(wav_fine)
f_fine = np.interp(wav_fine, model_w, model_f)
F2 = np.trapz(f_fine, wav_fine)
print(((F1 - F2)/F2)*100)

In [3]:
def calc_fbol(x, unit):
    ##########################################################
    # Function: calc_fbol                                    #
    # Inputs:                                                #
    #    star: star object                                   #
    #    x: sed object                                       #
    #    unit: unit string                                   #
    # Outputs:                                               #
    #    result: bolometric flux                             #
    #    error: error on bolometric flux                     #
    # How it works:                                          #
    #    1. Calls convert to generate the model values       #
    #    2. Defines a "model" for integration purposes       #
    #    3. Integrates the model                             #
    #    4. Sets bolometric flux value and error to star     #
    #    5. Returns values.                                  #
    ##########################################################
    _, _, _, _, model_w, model_f, _ = convert(x, unit)

    F1 = np.trapezoid(model_f, model_w)

    #testing the model's convergence
    wav_fine = np.linspace(model_w.min(), model_w.max(), 100*len(model_w))
    f_fine = np.interp(wav_fine, model_w, model_f)
    F2 = np.trapezoid(f_fine, wav_fine)

    convergence = ((F1-F2)/F2)*100
    if convergence >=0.1:
        print("Integration does not converge.")
    
    return F1

In [ ]:
d = star.dist
ra = star.ra_hms
dec = star.dec_dms

f = io.StringIO()
with contextlib.redirect_stdout(f):
    x = SEDFit(ra, dec, 1,use_gaia_params = False, use_gaia_xp = False, grid_type=model)

x.dist = d
#teff, logg, feh, av = initial_guess
x.addguesses(teff=teff, logg=logg, feh=feh, av = av)
x.addrange(teff = [4000,8000])

fbol = []
fbol_err = []
chi2s = []
chi2reds = []
seds = []

for i in range(num_iter):
    sed_obj = x
    print("Starting fit #", i+1)
    rand_sed = randomize_photometry(sed)
    downloadflux(sed_obj, rand_sed)
    #downloadflux(sed_obj, sed)
    set_quality(sed_obj)
    sed_obj.fit(use_gaia=False, idx=np.arange(0, len(sed_obj.sed['index'])), fitdist=False, fitteff=fitT, fitfeh=fit_feh, 
                fitlogg=fit_logg, fitav = fit_av, quality_check=False)
    
    numOparams = 1
    if fitT == True:
        numOparams += 1
    if fit_logg == True:
        numOparams += 1 
    if fit_feh == True:
        numOparams += 1
    if fit_av == True:
        numOparams +=1

    chi2, chi2r = chi2red(sed_obj, numOparams, verbose=False)
    chi2s.append(chi2)
    chi2reds.append(chi2r)

    fb = calc_fbol(sed_obj)
    fbol.append(fb)
    
    seds.append(sed_obj)

    print("Finished fit #", i+1)

In [ ]:
avg_fbol = np.mean(fbol)
std_fbol = np.std(fbol)
fbol_err = np.sqrt((std_fbol)**2 + (0.02*avg_fbol)**2)
precision = (fbol_err/avg_fbol)*100
print('Bolometric flux: ', round((avg_fbol/1e-8), 5), '+/-', round((fbol_err/1e-8), 5), 'x 10^{-8) erg/s/cm^2')
print('Precision: ', round(precision, 3), '%')

In [ ]:
def select_bestfit(star, sed_list, chi2list, chi2redlist):
    min_c2r = min(chi2redlist)
    minidx = np.argmin(chi2redlist)
    sed_bf = sed_list[minidx]
    minc2 = chi2list[minidx]
    star.SEDchi2 = minc2
    star.SEDchi2red = min_c2r
    return sed_bf

In [ ]:
sed_best = select_bestfit(star, seds, chi2s, chi2reds)

In [ ]:
x = sed_best
print("Distance: {} pc".format(x.getdist()))
print("AV: {} mag".format(x.getav()))
print("Radius: {} Rsun".format(x.getr()[0]))
print("Teff: {} K".format(x.getteff()[0]))
print("Log g: {} ".format(x.getlogg()[0]))
print("Fe/H: {}".format(x.getfeh()))

In [ ]:
if fitT == True:
    star.SEDTeff = x.getteff()[0]
if fit_logg == True:
    star.SEDlogg = x.getlogg()[0]
if fit_feh == True:
    star.SEDfeh = x.getfeh()
if fit_av == True:
    star.SEDAv = x.getav()

In [ ]:
x.sed

In [ ]:
plot_sed(sed_best, 'micron', logplot = True, fbol_lam = True, title = None, savefig = None, show = True)

In [ ]:
def fit_sed(sed, star, initial_guess, num_iter, model, teffrange=None, loggrange=None, fehrange=None, avrange = None, fitT=False, fit_logg=False,
            fit_feh=False, fit_av = False, verbose=False):
    d = star.dist
    ra = star.ra_hms
    dec = star.dec_dms

    f = io.StringIO()
    with contextlib.redirect_stdout(f):
        try:
            x = SEDFit(ra, dec, 1, use_gaia_params=False, use_gaia_xp = False, grid_type=model)
            x.addguesses(r=[1])
        except AttributeError:
            x = SEDFit(ra, dec, 0.5, use_gaia_params = False, use_gaia_xp = False, grid_type = model)

    teff, logg, feh, av = initial_guess
    x.dist = d
    x.addguesses(teff=teff, logg=logg, feh=feh, av = av)

    numOparams = 1
    if teffrange is not None:
        x.addrange(teff=teffrange)
        numOparams+=1
    if loggrange is not None:
        x.addrange(logg=loggrange)
        numOparams+=1
    if fehrange is not None:
        x.addrange(feh=fehrange)
        numOparams+=1
    if avrange is not None:
        x.addrange(av=avrange)
        numOparams+=1

    fbol = np.zeros(num_iter)
    chi2s = np.zeros(num_iter)
    chi2reds = np.zeros(num_iter)

    best_sed = None
    best_chi2red = np.inf
    best_idx = 0
    
    if not verbose:
        suppress_output = contextlib.redirect_stdout(io.StringIO())
    else:
        suppress_output = contextlib.nullcontext()
        
    for i in range(num_iter):
        #sed_obj = x
        #if verbose:
        print("Starting fit #", i+1)
        rand_sed = randomize_photometry(sed)
        downloadflux(x, rand_sed)
        #downloadflux(sed_obj, sed)
        #set_quality(sed_obj)
        if i == 0 or i % max(1, num_iter // 5) == 0:
            set_quality(x)
        with suppress_output:
            x.fit(use_gaia=False, idx=np.arange(0, len(x.sed['index'])), fitdist=False, fitteff=fitT, fitfeh=fit_feh, 
                  fitlogg=fit_logg, fitav = fit_av, quality_check=False)

        chi2, chi2r = chi2red(x, numOparams, verbose=False)
        chi2s[i] = chi2
        chi2reds[i] = chi2r

        fbol[i] = calc_fbol(x, unit)
        #fbol.append(fb)
        if chi2r < best_chi2red:
            best_chi2red = chi2r
            best_idx = i
            # Deep copy only the best SED to avoid reference issues
            import copy
            best_sed = copy.deepcopy(x)
        #seds.append(sed_obj)
        #if verbose:
        print("Finished fit #", i+1)
            
    avg_fbol = np.mean(fbol)
    std_fbol = np.std(fbol)
    fbol_err = np.sqrt((std_fbol)**2 + (0.02*avg_fbol)**2)
    precision = (fbol_err/avg_fbol)*100
    if verbose:
        print('Bolometric flux: ', round((avg_fbol/1e-8), 5), '+/-', round((fbol_err/1e-8), 5), 'x 10^{-8) erg/s/cm^2')
        print('Fbol precision: ', round(precision, 3), '%')
    star.fbol = round((avg_fbol/1e-8), 5)
    star.fbol_err = round((fbol_err/1e-8), 5)

    star.SEDchi2 = chi2s[best_idx]
    star.SEDchi2red = best_chi2red
    #sed_best = select_bestfit(star, seds, chi2s, chi2reds)
    
    if fitT == True:
        star.SEDTeff = best_sed.getteff()[0]
    if fit_logg == True:
        star.SEDlogg = best_sed.getlogg()[0]
    if fit_feh == True:
        star.SEDfeh = best_sed.getfeh()
    if fit_av == True:
        star.SEDAv = best_sed.getav()

    if verbose:
        print("Distance: {} pc".format(best_sed.getdist()))
        print("AV: {} mag".format(best_sed.getav()))
        print("Radius: {} Rsun".format(best_sed.getr()[0]))
        print("Teff: {} K".format(best_sed.getteff()[0]))
        print("Log g: {} ".format(best_sed.getlogg()[0]))
        print("Fe/H: {}".format(best_sed.getfeh()))

    return best_sed

In [4]:
from numpy.random import default_rng
def randomize_photometry(sed):
    """Ultra-fast version using numpy's new RNG"""
    rng = default_rng()  # Faster than np.random.normal
    
    new_sed = sed.copy()
    rand_f = rng.normal(new_sed['flux'], new_sed['eflux'])
    new_sed['flux'] = rand_f
    
    return new_sed

In [ ]:
def single_iteration(iter_num, sed, star, initial_guess, unit, model, ranges, fit_params, numOparams):
    # Each worker creates its own SEDFit object
    d = star.dist
    ra = star.ra_hms
    dec = star.dec_dms
        
    x = SEDFit(ra, dec, 1, use_gaia_params=False, use_gaia_xp=False, grid_type=model)
    x.dist = d
        
    teff, logg, feh, av = initial_guess
    x.addguesses(teff=teff, logg=logg, feh=feh, av=av, r=[1])
        
        # Apply ranges
    for param, rng in ranges.items():
        x.addrange(**{param: rng})
        
        # Randomize and fit
    rand_sed = randomize_photometry(sed)
    downloadflux(x, rand_sed)
        
    if iter_num == 0:
        set_quality(x)
        
    x.fit(use_gaia=False, idx=np.arange(0, len(x.sed['index'])),fitdist=False, quality_check=False, **fit_params)
        
    chi2, chi2r = chi2red(x, numOparams, verbose=False)
    fb = calc_fbol(x, unit)
        
    return {'chi2': chi2, 'chi2r': chi2r, 'fbol': fb, 'sed': x}

In [ ]:
from multiprocessing import Pool
from functools import partial

def fit_sed_parallel(sed, star, initial_guess, num_iter, unit, model, teffrange=None, 
                     loggrange=None, fehrange=None, avrange=None, fitT=False, 
                     fit_logg=False, fit_feh=False, fit_av=False, n_jobs=4, verbose=False):
    """
    Parallel version using multiprocessing
    """
    
    # Prepare parameters
    ranges = {}
    if teffrange is not None:
        ranges['teff'] = teffrange
    if loggrange is not None:
        ranges['logg'] = loggrange
    if fehrange is not None:
        ranges['feh'] = fehrange
    if avrange is not None:
        ranges['av'] = avrange
    
    fit_params = {'fitteff': fitT, 'fitlogg': fit_logg, 'fitfeh': fit_feh, 'fitav': fit_av}
    numOparams = 1 + sum([fitT, fit_logg, fit_feh, fit_av])
    
    # Run parallel iterations
    partial_func = partial(single_iteration, sed=sed, star=star, initial_guess=initial_guess,
                          unit=unit, model=model, ranges=ranges, fit_params=fit_params,
                          numOparams=numOparams)
    
    with Pool(n_jobs) as pool:
        results = pool.map(partial_func, range(num_iter))
    
    # Process results
    fbol = np.array([r['fbol'] for r in results])
    chi2reds = np.array([r['chi2r'] for r in results])
    chi2s = np.array([r['chi2'] for r in results])
    
    best_idx = np.argmin(chi2reds)
    best_sed = results[best_idx]['sed']
    
    # Calculate statistics
    avg_fbol = np.mean(fbol)
    std_fbol = np.std(fbol)
    fbol_err = np.sqrt((std_fbol) ** 2 + (0.02 * avg_fbol) ** 2)
    
    star.fbol = round((avg_fbol / 1e-8), 5)
    star.fbol_err = round((fbol_err / 1e-8), 5)
    star.SEDchi2 = chi2s[best_idx]
    star.SEDchi2red = chi2reds[best_idx]
    
    if fitT:
        star.SEDTeff = best_sed.getteff()[0]
    if fit_logg:
        star.SEDlogg = best_sed.getlogg()[0]
    if fit_feh:
        star.SEDfeh = best_sed.getfeh()
    if fit_av:
        star.SEDAv = best_sed.getav()
    
    return best_sed

In [5]:
initial_guess = [5000,4.31,0.19,0]
model = 'phoenix'
sed = phot_data
num_iter = 500
fitT = True
fit_feh = False
fit_logg = False
fit_av = False
unit = 'AA'

In [ ]:
import os
print(f'CPU cores available: {os.cpu_count()}')

In [ ]:
from joblib import Parallel, delayed
import time

def fit_sed_parallel_debug(sed, star, initial_guess, num_iter, unit, model, 
                           teffrange=None, loggrange=None, fehrange=None, avrange=None,
                           fitT=False, fit_logg=False, fit_feh=False, fit_av=False, 
                           n_jobs=-2, verbose=False, timeout=None):
    """
    Parallel version with progress monitoring and timeout
    """
    
    def single_iteration(iter_num):
        start_time = time.time()
        print(f"Starting iteration {iter_num}...", flush=True)
        
        d = star.dist
        ra = star.ra_hms
        dec = star.dec_dms
        
        # Suppress output
        with contextlib.redirect_stdout(io.StringIO()):
            try:
                x = SEDFit(ra, dec, 1, use_gaia_params=False, use_gaia_xp=False, grid_type=model)
                x.addguesses(r=[1])
            except AttributeError:
                x = SEDFit(ra, dec, 0.5, use_gaia_params=False, use_gaia_xp=False, grid_type=model)
        
        #print(f"  Iter {iter_num}: SEDFit created ({time.time()-start_time:.1f}s)", flush=True)
        
        x.dist = d
        teff, logg, feh, av = initial_guess
        x.addguesses(teff=teff, logg=logg, feh=feh, av=av)
        
        if teffrange is not None:
            x.addrange(teff=teffrange)
        if loggrange is not None:
            x.addrange(logg=loggrange)
        if fehrange is not None:
            x.addrange(feh=fehrange)
        if avrange is not None:
            x.addrange(av=avrange)
        
        #print(f"  Iter {iter_num}: Randomizing photometry ({time.time()-start_time:.1f}s)", flush=True)
        rand_sed = randomize_photometry(sed)
        
        #print(f"  Iter {iter_num}: Downloading flux ({time.time()-start_time:.1f}s)", flush=True)
        downloadflux(x, rand_sed)
        
        if iter_num == 0:
            #print(f"  Iter {iter_num}: Running quality check ({time.time()-start_time:.1f}s)", flush=True)
            set_quality(x)
        
        print(f"  Iter {iter_num}: Starting fit ({time.time()-start_time:.1f}s)", flush=True)
        with contextlib.redirect_stdout(io.StringIO()):
            x.fit(use_gaia=False, idx=np.arange(0, len(x.sed['index'])), 
                  fitdist=False, fitteff=fitT, fitfeh=fit_feh,
                  fitlogg=fit_logg, fitav=fit_av, quality_check=False)
        
        #print(f"  Iter {iter_num}: Calculating chi2 ({time.time()-start_time:.1f}s)", flush=True)
        numOparams = 1 + sum([fitT, fit_logg, fit_feh, fit_av])
        chi2, chi2r = chi2red(x, numOparams, verbose=False)
        
        #print(f"  Iter {iter_num}: Calculating fbol ({time.time()-start_time:.1f}s)", flush=True)
        fb = calc_fbol(x, unit)
        
        elapsed = time.time() - start_time
        print(f"✓ Iter {iter_num} completed in {elapsed:.1f}s", flush=True)
        
        return {'chi2': chi2, 'chi2r': chi2r, 'fbol': fb, 'sed': x}
    
    print(f"Starting parallel processing with {n_jobs} jobs...")
    
    # Use verbose mode to see progress
    results = Parallel(n_jobs=n_jobs, verbose=10, timeout=timeout)(
        delayed(single_iteration)(i) for i in range(num_iter)
    )
    
    print("All iterations completed!")
    
    # Process results
    fbol = np.array([r['fbol'] for r in results])
    chi2reds = np.array([r['chi2r'] for r in results])
    chi2s = np.array([r['chi2'] for r in results])
    
    best_idx = np.argmin(chi2reds)
    best_sed = results[best_idx]['sed']
    
    # Calculate statistics
    avg_fbol = np.mean(fbol)
    std_fbol = np.std(fbol)
    fbol_err = np.sqrt((std_fbol) ** 2 + (0.02 * avg_fbol) ** 2)
    
    star.fbol = round((avg_fbol / 1e-8), 5)
    star.fbol_err = round((fbol_err / 1e-8), 5)
    star.SEDchi2 = chi2s[best_idx]
    star.SEDchi2red = chi2reds[best_idx]
    
    if fitT:
        star.SEDTeff = best_sed.getteff()[0]
    if fit_logg:
        star.SEDlogg = best_sed.getlogg()[0]
    if fit_feh:
        star.SEDfeh = best_sed.getfeh()
    if fit_av:
        star.SEDAv = best_sed.getav()
    
    return best_sed

In [11]:
from joblib import Parallel, delayed
import time

def fit_sed_optimized_debug(sed, star, initial_guess, num_iter, unit, model, 
                            teffrange=None, loggrange=None, fehrange=None, avrange=None,
                            fitT=False, fit_logg=False, fit_feh=False, fit_av=False, 
                            verbose=False, debug = False):
    """
    Serial version with progress monitoring
    """
    import time
    
    d = star.dist
    ra = star.ra_hms
    dec = star.dec_dms
    if verbose:
        print("Initializing SEDFit object...")
    with contextlib.redirect_stdout(io.StringIO()):
        try:
            x = SEDFit(ra, dec, 1, use_gaia_params=False, use_gaia_xp=False, grid_type=model)
            x.addguesses(r=[1])
        except AttributeError:
            x = SEDFit(ra, dec, 0.5, use_gaia_params=False, use_gaia_xp=False, grid_type=model)

    teff, logg, feh, av = initial_guess
    x.dist = d
    x.addguesses(teff=teff, logg=logg, feh=feh, av=av)
    
    if teffrange is not None:
        x.addrange(teff=teffrange)
    if loggrange is not None:
        x.addrange(logg=loggrange)
    if fehrange is not None:
        x.addrange(feh=fehrange)
    if avrange is not None:
        x.addrange(av=avrange)

    numOparams = 1 + sum([fitT, fit_logg, fit_feh, fit_av])

    fbol = np.empty(num_iter)
    chi2reds = np.empty(num_iter)
    
    best_chi2red = np.inf
    best_sed = None

    print(f"Starting {num_iter} iterations...")
    overall_start = time.time()
    
    with contextlib.redirect_stdout(io.StringIO()):
        for i in range(num_iter):
            iter_start = time.time()
            
            # Print progress every 10 iterations or for first few
            if i < 3 or i % 10 == 0:
                print(f"\n[{i+1}/{num_iter}] Starting iteration...", flush=True)
            
            rand_sed = randomize_photometry(sed)
            downloadflux(x, rand_sed)
            
            if i == 0:
                set_quality(x)
            
            x.fit(use_gaia=False, idx=np.arange(0, len(x.sed['index'])), 
                  fitdist=False, fitteff=fitT, fitfeh=fit_feh,
                  fitlogg=fit_logg, fitav=fit_av, quality_check=False)

            _, chi2r = chi2red(x, numOparams, verbose=False)
            chi2reds[i] = chi2r
            fbol[i] = calc_fbol(x, unit)

            if chi2r < best_chi2red:
                best_chi2red = chi2r
                import copy
                best_sed = copy.deepcopy(x)
            
            iter_time = time.time() - iter_start
            
            if i < 3 or i % 10 == 0:
                elapsed = time.time() - overall_start
                avg_time = elapsed / (i + 1)
                remaining = avg_time * (num_iter - i - 1)
                if debug:
                    print(f"[{i+1}/{num_iter}] Complete in {iter_time:.1f}s | "
                          f"Avg: {avg_time:.1f}s | ETA: {remaining/60:.1f} min", flush=True)
    if debug:
        total_time = time.time() - overall_start
        print(f"\n✓ All {num_iter} iterations complete in {total_time/60:.1f} minutes or {total_time:.3f} seconds")

    # Calculate statistics
    avg_fbol = np.mean(fbol)
    std_fbol = np.std(fbol)
    fbol_err = np.sqrt(std_fbol**2 + (0.02 * avg_fbol)**2)
    
    star.fbol = round((avg_fbol / 1e-8), 5)
    star.fbol_err = round((fbol_err / 1e-8), 5)
    star.SEDchi2red = best_chi2red

    if fitT:
        star.SEDTeff = best_sed.getteff()[0]
    if fit_logg:
        star.SEDlogg = best_sed.getlogg()[0]
    if fit_feh:
        star.SEDfeh = best_sed.getfeh()
    if fit_av:
        star.SEDAv = best_sed.getav()

    return best_sed

In [20]:
#if __name__ == '__main__':
    # Use the debug version with just 5 iterations
result = fit_sed_optimized_debug(sed, star, [5000, 4.5, 0.0, 0.1], 
                                 num_iter=5,  # Start small
                                 unit='AA', 
                                 model='phoenix',
                                 verbose=True)

Initializing SEDFit object...
Starting 5 iterations...

✓ All 5 iterations complete in 0.1 minutes or 6.579 seconds


index,sed_filter,la,width,flux,eflux,model
,,Angstrom,Angstrom,,erg / (Angstrom s cm2),
int64,str17,float64,float64,float64,float64,float64
0,Johnson.U,3511.89,328.5,-7.917813856084507,0.058237200454887804,-8.078789663992524
1,TYCHO.TYCHO.B_MvB,4194.96,370.695,-7.510309178389956,0.022513133845333523,-7.633662455085301
2,Johnson.B,4382.77,505.85,-7.413493807103604,0.0219343592986413,-7.5343037415658705
3,GAIA.GAIA3.Gbp,5109.71,1078.75,-7.365289113332223,0.021733109483493523,-7.40916706189392
4,TYCHO.TYCHO.V_MvB,5300.19,566.775,-7.319745740213824,0.02206509254745958,-7.347650539113314
5,Johnson.V,5501.4,444.9,-7.280896797227464,0.02187965656660768,-7.332102468028833
6,GAIA.GAIA3.G,6217.59,2026.485,-7.338951596880335,0.021728844871176007,-7.327267835775147
7,GAIA.GAIA3.Grp,7769.02,1462.22,-7.324687659035937,0.021753097890601168,-7.260023881921914


In [ ]:
if __name__ == '__main__':
    import time
    
    print("Starting test...")
    start = time.time()
    
    result = fit_sed_parallel_debug(
        sed, star, [5000, 4.5, 0.0, 0.1], 
        num_iter=10,  # Just 1 iteration
        unit='AA', 
        model='phoenix',
        n_jobs=4,  # Serial first
        verbose=True
    )
    
    print(f"\nTotal time: {time.time() - start:.1f} seconds")

In [ ]:
test = fit_sed_parallel(sed, star, initial_guess, num_iter, unit, model, teffrange=[4000,8000], fitT=fitT, fit_logg=fit_logg, 
               fit_feh=fit_feh, fit_av = fit_av, verbose=True)

In [ ]:
test = fit_sed(sed, star, initial_guess, num_iter, model, teffrange=[4000,8000], fitT=fitT, fit_logg=fit_logg, 
               fit_feh=fit_feh, fit_av = fit_av, verbose=True)

In [ ]:
test.sed

In [ ]:
plot_sed(test, 'micron', logplot = True, fbol_lam = True, title = None, savefig = None, show = True)